In [ ]:
import pandas as pd
import MySQLdb
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn')

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


db = client.get_database('telematics_db')






In [ ]:
vehicle_list=list(pd.read_sql("select vehicle_id from operator_vehicles where deboarding_time is null ",con=mysql_vds)['vehicle_id'])


In [ ]:

count=0


In [ ]:
def blacklist(vid):
    #print("entered")
    global count
    count+=1
    print("-"+str(count)+"-")
    print("  ><  ")
    try:
        
        data=pd.read_sql("select vehicle_id,day,jump_percent from device_health where vehicle_id={} and  day<{}".format(vid,str("'"+str(pd.datetime.today().date())+"'")),con=engine)                       
        data.drop_duplicates(['day','vehicle_id'],inplace=True)
        data=data[data['jump_percent']!=-1]
        jump_mean=data['jump_percent'].mean()
        number_of_days=len(data)
        row=[vid,jump_mean,number_of_days]
    
    except Exception as e:
        print("WENT INTO EXCPET")
        print(e)
        row=[vid,-1,-1]
    return row
    
    
    
    
    

In [ ]:
from multiprocessing.dummy import Pool as ThreadPool
pool = ThreadPool(200)
reuslts=pool.map(blacklist,vehicle_list)
pool.close()
pool.join()


In [ ]:
complete_blacklist_data=pd.DataFrame(reuslts,columns=['vehicle_id','jump_percent','number_of_days'])

In [ ]:
complete_blacklist_data.number_of_days.value_counts()

In [ ]:
three_plus_days=complete_blacklist_data[complete_blacklist_data['number_of_days']>3]

In [ ]:
len(three_plus_days)

In [ ]:
vehicle_details=pd.read_sql("select * from vehicle_details",con=engine
                           )

In [ ]:
analysis=pd.merge(three_plus_days,vehicle_details,on='vehicle_id',how='left')

In [ ]:
blacklist_devices=analysis[analysis['jump_percent']>20]

In [ ]:
blacklist_devices.to_csv('blacklist_devices.csv')

In [ ]:
blacklist_devices

In [ ]:
plt.hist(data[data['number_of_days']>3][data['jump_percent']>20]['vehicle_id'],bins=range(0,90000,1000))

In [ ]:
plt.hist(data['number_of_days'])

In [ ]:
plt.hist(data[data['jump_percent']>20][data['number_of_days']>2]['vehicle_id'].unique(),bins=range(0,80000,100))
plt.xlabel("vehicle_id range")
plt.ylabel("number of vehicles above 20% jump")

In [ ]:
len(data[data['jump_percent']>20]['vehicle_id'].unique())/len(data.dropna()['vehicle_id'].unique())

In [ ]:
data.dropna()